# Bank Customer Churn Analysis: Predictive Modeling

## Business context
The descriptive analysis (01_exploration.ipynb) showed which customer segments
churn most: German customers, the 50-59 age band, inactive members, and others.
But each cut looked at one factor in isolation, so none can say whether a factor
predicts churn on its own or only appears to because it travels with others. This
notebook builds a logistic regression model to separate the two.

## Question
What factors most strongly predict churn, holding the other factors constant?

## What this notebook does
- Prepares the data for modeling by encoding the categorical features
- Fits a logistic regression with statsmodels and reads the coefficients as odds
  ratios: which factors raise or lower the odds of churn, and by how much
- Evaluates the model as a classifier (confusion matrix, precision, recall, ROC-AUC)
  and addresses the class imbalance the descriptive work flagged
- Translates the results into retention recommendations

## Tools
- Python 3.12, pandas, numpy
- statsmodels (logistic regression and inference)
- scikit-learn (train/test split, evaluation metrics)
- matplotlib, seaborn

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/Churn_Modelling.csv')
print("Shape:", df.shape)
df.head()

Shape: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [8]:
# --- Feature engineering ---

# Drop identifier columns: they carry no predictive information.
model_df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname']).copy()

# Collapse NumOfProducts into 1, 2, and 3+.
# Two reasons. (1) Churn is non-monotonic across product counts (your products
# cut), so a single linear term would average the dip and the spike into nothing;
# treating products as categories lets the model capture the real shape. (2) The
# 4-product group churned at exactly 100% in the data, which causes "perfect
# separation" and breaks logistic regression's coefficient estimation. Merging 3
# and 4 into "3+" removes the 100% cell while keeping the high-risk signal.
model_df['ProductGroup'] = np.where(
    model_df['NumOfProducts'] >= 3, '3+',
    model_df['NumOfProducts'].astype(str)
)
model_df = model_df.drop(columns=['NumOfProducts'])

# One-hot encode the categorical features. drop_first drops one level per feature
# as the reference baseline (France for geography, Female for gender, 1 product
# for product group); each remaining dummy is then read relative to that baseline.
# dtype=int because statsmodels expects numeric, not boolean, columns.
model_df = pd.get_dummies(
    model_df,
    columns=['Geography', 'Gender', 'ProductGroup'],
    drop_first=True,
    dtype=int
)

print("Shape:", model_df.shape)
print("Columns:", model_df.columns.tolist())
model_df.head()

Shape: (10000, 13)
Columns: ['CreditScore', 'Age', 'Tenure', 'Balance', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_Germany', 'Geography_Spain', 'Gender_Male', 'ProductGroup_2', 'ProductGroup_3+']


,CreditScore,Age,Tenure,Balance,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male,ProductGroup_2,ProductGroup_3+
0,619,42,2,0.00,1,1,101348.88,1,0,0,0,0,0
1,608,41,1,83807.86,0,1,112542.58,0,0,1,0,0,0
2,502,42,8,159660.80,1,0,113931.57,1,0,0,0,0,1
3,699,39,1,0.00,0,0,93826.63,0,0,0,0,1,0
4,850,43,2,125510.82,1,1,79084.10,0,0,1,0,0,0


In [9]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=['Exited'])
y = model_df['Exited']

# Stratified split: stratify=y holds the ~20% churn proportion identical in train
# and test. That matters with imbalanced classes; a random split could deal the
# test set a different churn rate and distort the evaluation. random_state fixes
# the split so the notebook reproduces.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "| churn rate:", round(y_train.mean(), 3))
print("Test: ", X_test.shape, "| churn rate:", round(y_test.mean(), 3))

Train: (8000, 12) | churn rate: 0.204
Test:  (2000, 12) | churn rate: 0.204


In [10]:
from sklearn.preprocessing import StandardScaler

# Scale only the continuous features. The 0/1 columns (binary flags and dummies)
# are left alone: standardizing them would not help and would make their
# coefficients harder to read. The scaler is fit on the training data only, then
# applied to both sets, so nothing about the test set's distribution leaks into
# the training data.
continuous = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous] = scaler.fit_transform(X_train[continuous])
X_test_scaled[continuous] = scaler.transform(X_test[continuous])

X_train_scaled.head()

,CreditScore,Age,Tenure,Balance,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male,ProductGroup_2,ProductGroup_3+
2151,1.058568,1.715086,0.684723,-1.226059,1,0,1.042084,0,0,1,0,0
8392,0.913626,-0.659935,-0.696202,0.413288,1,0,-0.623556,1,0,1,0,0
5006,1.079274,-0.184931,-1.731895,0.601687,1,1,0.308128,1,0,0,1,0
4117,-0.929207,-0.184931,-0.005739,-1.226059,1,0,-0.290199,0,0,1,1,0
7182,0.427035,0.955079,0.339492,0.548318,0,1,0.135042,1,0,1,1,0


In [11]:
import statsmodels.api as sm

# statsmodels does not add an intercept by default, so add a constant column.
X_train_const = sm.add_constant(X_train_scaled)

# Fit the logistic regression by maximum likelihood.
logit_model = sm.Logit(y_train, X_train_const).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.373390
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                 Exited   No. Observations:                 8000
Model:                          Logit   Df Residuals:                     7987
Method:                           MLE   Df Model:                           12
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:                  0.2614
Time:                        11:25:51   Log-Likelihood:                -2987.1
converged:                       True   LL-Null:                       -4044.5
Covariance Type:            nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.5892      0.085     -6.951      0.000      -0.755      -0.423
CreditSc

In [12]:
# Odds ratio > 1: the factor raises the odds of churn. < 1: it lowers them.
odds = pd.DataFrame({
    'coef': logit_model.params,
    'odds_ratio': np.exp(logit_model.params),
    'p_value': logit_model.pvalues,
})
odds['ci_low'] = np.exp(logit_model.conf_int()[0])
odds['ci_high'] = np.exp(logit_model.conf_int()[1])
odds.round(3)

,coef,odds_ratio,p_value,ci_low,ci_high
const,-0.589,0.555,0.000,0.470,0.655
CreditScore,-0.081,0.922,0.013,0.865,0.983
Age,0.725,2.064,0.000,1.938,2.198
Tenure,-0.022,0.979,0.505,0.918,1.043
Balance,-0.044,0.957,0.270,0.885,1.035
HasCrCard,-0.114,0.892,0.110,0.776,1.026
IsActiveMember,-1.058,0.347,0.000,0.303,0.398
EstimatedSalary,0.044,1.045,0.179,0.980,1.115
Geography_Germany,0.996,2.707,0.000,2.308,3.174
Geography_Spain,0.082,1.085,0.333,0.919,1.281


**Age, German geography, activity status, and product holdings are the strongest independent churn predictors, while balance and salary wash out after controls.**

The logistic regression has meaningful explanatory power, with a pseudo R-squared of 0.26, and the coefficient table is suitable for interpretation through odds ratios. The strongest independent churn signals are product holdings, age, geography, and activity status. Customers with three or more products have about 16 times the odds of churning compared with one-product customers, holding all other variables constant. Two-product customers move in the opposite direction, with only about 0.21 times the odds of churn compared with one-product customers, confirming the descriptive finding that two-product holders are unusually sticky.

Several of the strongest descriptive findings also survive the model. Age remains one of the cleanest predictors: each standard-deviation increase in age, roughly a decade, is associated with about double the odds of churn. German geography also remains highly predictive, with German customers having about 2.7 times the odds of churn compared with French customers, even after controlling for age, balance, activity, product count, and the other variables. Activity status is also a strong signal in the protective direction: active members have about 0.35 times the odds of churn after controls. Gender also matters in the model, with male customers having about 0.61 times the odds of churn compared with female customers. Credit score is statistically significant but much weaker, with only a small protective effect.

The most important modeling insight is what disappears after controls. Balance, estimated salary, tenure, and credit-card ownership are not statistically significant once the other factors are included. This is especially important for balance: the descriptive notebook showed visible churn differences by balance band, but the regression suggests that balance itself is not an independent churn predictor after accounting for customer profile, geography, activity, and product holdings. In other words, the apparent balance signal was likely acting partly as a proxy for other customer differences.

For the bank, this shifts the interpretation from broad segment comparison to independent prediction. The strongest churn predictors are not mainly money-size variables such as balance or salary, but customer profile and engagement variables: age, geography, activity, gender, and product structure. This supports targeting retention around engagement and customer-risk profiles rather than assuming that high account balance alone explains churn risk.

**Caveats**

* The 3+ product effect is very large and statistically significant, but its confidence interval is wide. The model confirms that the effect is real, but the exact “16 times” multiple should be treated cautiously because the group is relatively small.
* These are predictive associations in an observational snapshot. The model shows which factors are associated with churn after controls, but it does not prove that changing a factor, such as activity status or product count, would directly cause churn risk to change.
